# 建立影像生成應用程式（Azure OpenAI）

大型語言模型不只是文字生成，你亦可以從文字描述生成影像。影像作為一種感官模態，在醫療科技、建築、旅遊、遊戲開發、行銷等領域都有廣泛應用。在本課程中，我們將使用 Microsoft Foundry 上的最新 **GPT 影像** 模型來實作。

## 學習目標

完成本課程後，你將能：

- 解釋什麼是影像生成及其應用範圍。
- 了解 `gpt-image` 模型系列及其與傳統 DALL·E 模型的不同。
- 建立影像生成應用程式並進行影像編輯。

## 什麼是影像生成？

影像生成模型可依照文字提示創造出影像。像 `gpt-image` 這類現代模型在訓練中學習文字與影像之間的關係，然後從隨機噪聲逐步生成符合你提示的影像。

兩個知名的影像模型系列有：

- **`gpt-image`（OpenAI）** — 本課程使用的現代世代模型。支援文字轉影像和影像編輯（帶遮罩修復）。
- **Midjourney** — 一個知名的第三方模型，具備獨立服務及基於 Discord 的工作流程。

> 較早期的OpenAI影像模型 — **DALL·E 2** 和 **DALL·E 3** — 已屬舊有產品。DALL·E 3 不再支援新部署，且 `create_variation` 這類功能只存於 DALL·E 2。新應用請使用 `gpt-image` 模型。

在 Microsoft Foundry 上，**`gpt-image-2`** 是最新且功能最強大的影像模型，為推薦預設。`gpt-image-1.5` 和 `gpt-image-1-mini` 也已普遍提供。

> **重要提醒：** `gpt-image` 模型回傳的生成影像是以 **base64** 格式（`b64_json`），而非 URL。你的程式碼需要解碼 base64 字串成為位元組並保存 — 這裡並無影像下載的 URL。


## 建立你的第一個圖像生成應用程式

那麼，建立一個圖像生成應用程式需要什麼？你需要以下的函式庫：

- **python-dotenv**，強烈建議使用此函式庫，將你的密鑰保存在 *.env* 檔案中，避免寫在程式碼裡。
- **openai**，這個函式庫用來與 OpenAI API 互動。
- **pillow**，在 Python 中操作圖像。

如果尚未完成，請依照[Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) 頁面上的指示建立 Microsoft Foundry 資源和模型。選擇 **gpt-image-2** 作為模型（最新的 Azure OpenAI 圖像模型；DALL·E 是舊版）。

1. 建立一個 *.env* 檔案，內容如下：

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    請在 Microsoft Foundry 入口網站中你資源的「部署」區段找到此資訊。



1. 將以上的庫收集到名為 *requirements.txt* 的文件中，如下：

    ```text
    python-dotenv
    openai
    pillow
    ```


1. 接著，建立虛擬環境並安裝這些庫：


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> 對於 Windows，請使用以下命令來建立並啟動您的虛擬環境：

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. 在名為 *app.py* 的檔案中加入以下程式碼：

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # 匯入 dotenv
    dotenv.load_dotenv()

    # 配置 Azure OpenAI（Microsoft Foundry）用戶端。
    # 影像模型需要較新的 API 版本 — 請查看 Foundry 文件了解您的模型所需的版本。
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # 使用影像生成 API 建立影像
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此輸入您的提示文字
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # 例如 "gpt-image-2"
        )
        # 設定儲存影像的目錄
        image_dir = os.path.join(os.curdir, 'images')

        # 若該目錄不存在，則建立它
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # 初始化影像路徑（注意檔案格式應為 png）
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image 模型以 base64（b64_json）格式回傳影像，而非 URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # 在預設影像檢視器中顯示影像
        image = Image.open(image_path)
        image.show()

    # 捕捉例外狀況
    except BadRequestError as err:
        print(err)

    ```

讓我們來解釋這段程式碼：

- 首先，我們匯入需要的函式庫，包括 OpenAI 函式庫、dotenv 函式庫、base64 模組和 Pillow 函式庫。

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- 接著，我們從 *.env* 檔案載入環境變數。

    ```python
    # 導入 dotenv
    dotenv.load_dotenv()
    ```

- 然後，我們配置 Azure OpenAI（Microsoft Foundry）用戶端。

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- 接著，我們生成圖片：

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此輸入你的提示文字
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` 模型會將圖片作為 **base64** 字串回傳於 `data[0].b64_json`。我們將其解碼為位元組並寫入檔案 — 這裡沒有下載用的 URL。

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- 最後，我們開啟圖片並使用標準圖片檢視器顯示它：

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### 有關生成圖片的更多細節

讓我們來看看 `images.generate` 的參數：

- **prompt**，是用來生成圖片的文字提示。這裡是「Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils」。
- **size**，是生成圖片的大小（例如 `1024x1024`、`1536x1024`、`1024x1536`，或 `"auto"`）。
- **n**，是生成圖片的數量。這裡我們生成一張。
- **model**，是您的圖片模型部署名稱（例如 `gpt-image-2`）。

> 影像模型並不接受 `temperature` 參數 — 那是文字生成的控制。若要獲得多樣性，請再次呼叫 API；要減少多樣性，請使您的提示更具體。

## 圖像生成的其他功能

您已經看過如何使用少量 Python 代碼產生一張圖片。`gpt-image` 模型也可以 <strong>編輯</strong> 現有的圖片。透過提供一張圖片、一個可選的 **mask** （標示欲更改區域）和提示文字，您可以修改圖片的部分內容 — 例如，為我們的小兔子加上一頂帽子。

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# 編輯結果亦會以 base64 形式返回
edited_image = base64.b64decode(response.data[0].b64_json)
```

基礎圖片只包含兔子；最終圖片則在兔子上加了帽子。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
